# 03 — Hallazgos

Los números que se cuentan en dos minutos.

**Todo lo que sigue compara precio por kilo, por litro o por unidad**, nunca
precio de envase. Sin esa normalización, "la cerveza más barata" sería una lata
de 269 ml y "la más cara" un barril de 50 litros.

In [ ]:
import os, sys, warnings
sys.path.insert(0, "..")
# Los datos crudos viven fuera del repo (pesan GB). Si moviste la carpeta,
# cambiá esta ruta o exportá RADAR_DATA_DIR antes de abrir el notebook.
os.environ.setdefault("RADAR_DATA_DIR", os.path.expanduser("~/sepa-data"))
warnings.filterwarnings("ignore")

import duckdb, pandas as pd, matplotlib.pyplot as plt
from src import config as cfg

pd.set_option("display.max_columns", 40); pd.set_option("display.width", 150)
plt.rcParams.update({"figure.figsize": (11, 4.5), "axes.grid": True, "grid.alpha": .25,
                     "axes.spines.top": False, "axes.spines.right": False,
                     "axes.titleweight": "bold"})

PARQUET = str(cfg.INTERIM / "fecha=*" / "*.parquet")
con = duckdb.connect()
dias = sorted(p.name.split("=")[1] for p in cfg.INTERIM.glob("fecha=*"))
assert dias, f"No hay datos en {cfg.INTERIM}. Corré primero: python -m src.clean"
print(f"Datos: {cfg.DATA}")
print(f"{len(dias)} días — de {dias[0]} a {dias[-1]}")

import json
disp   = pd.read_csv(cfg.PROCESSED / "fact_dispersion.csv", parse_dates=["fecha"])
canast = pd.read_csv(cfg.PROCESSED / "fact_canasta.csv", parse_dates=["fecha"])
rank   = pd.read_csv(cfg.PROCESSED / "fact_ranking_cadenas.csv", parse_dates=["fecha"])
precio = pd.read_csv(cfg.PROCESSED / "fact_precio_item.csv", parse_dates=["fecha"])
dim    = pd.read_csv(cfg.PROCESSED / "dim_item.csv")
H      = json.load(open(cfg.PROCESSED / "resumen_hallazgos.json"))

pesos = lambda v: f"${v:,.0f}".replace(",", ".")
ULT = disp.fecha.max()
print("Última fecha:", ULT.date())

## Hallazgo 1 — El mismo producto, precios muy distintos

Comparo el precio por unidad de cada item **dentro de la misma provincia**, entre
cadenas. Comparar entre provincias mezclaría costos logísticos e impuestos
provinciales, que son diferencias reales pero de otra naturaleza.

In [ ]:
ult = disp[disp.fecha == ULT]
top = (ult.groupby("item")
          .agg(brecha_pct=("gap_pct", "median"),
               p_min=("precio_min", "median"), p_max=("precio_max", "median"),
               provincias=("provincia", "nunique"))
          .join(dim.set_index("item")["unidad_base"])
          .sort_values("brecha_pct", ascending=False))
display(top.round(1))

print(f"Brecha mediana de toda la canasta: {ult.gap_pct.median():.1f}%")
print(f"Rango entre items: {top.brecha_pct.min():.0f}% a {top.brecha_pct.max():.0f}%")

In [ ]:
# Traducido a plata, producto por producto
t = top.head(12).iloc[::-1]
fig, ax = plt.subplots(figsize=(10, 6))
ax.barh(t.index, t.brecha_pct, color="#1f4e79")
for i, (v, lo, hi, u) in enumerate(zip(t.brecha_pct, t.p_min, t.p_max, t.unidad_base)):
    ax.text(v + 1, i, f"  {pesos(lo)} → {pesos(hi)} por {u}", va="center", fontsize=8.5, color="#444")
ax.set_xlim(0, t.brecha_pct.max() * 1.55)
ax.set_xlabel("Diferencia entre la cadena más cara y la más barata (%)")
ax.set_title("El mismo producto, la misma provincia");

### ¿La dispersión es igual en todas las categorías?

Hipótesis razonable: los productos frescos y de marca propia deberían dispersar
más que los de marca líder, donde el precio está más anclado.

In [ ]:
cat = (ult.groupby("categoria")
          .agg(brecha_mediana=("gap_pct", "median"),
               items=("item", "nunique"), observaciones=("gap_pct", "size"))
          .sort_values("brecha_mediana", ascending=False))
display(cat.round(1))

ax = cat.brecha_mediana.iloc[::-1].plot.barh(color="#1f4e79")
ax.set_xlabel("Brecha mediana (%)"); ax.set_title("Dispersión por categoría");

## Hallazgo 2 — Cuánto cuesta elegir mal

Traducido a plata: cuánto gasta por mes un hogar de 4 personas según dónde compre.

Los tres escenarios:

- **Óptimo**: cada producto en la cadena donde está más barato. No es alcanzable
  en la práctica —nadie recorre 18 cadenas— pero define el techo del ahorro.
- **Típico**: todo a precio mediano de mercado.
- **Peor**: todo en la cadena más cara.

In [ ]:
c = canast.groupby("fecha")[["canasta_optima", "canasta_tipica", "canasta_peor"]].mean()
display(c.round(0))

u = c.iloc[-1]
print(f"\nCanasta a precio de mercado:  {pesos(u.canasta_tipica)}")
print(f"Comprando siempre barato:      {pesos(u.canasta_optima)}")
print(f"Diferencia:                    {pesos(u.canasta_tipica - u.canasta_optima)}/mes "
      f"({100*(1-u.canasta_optima/u.canasta_tipica):.1f}%)")
print(f"Entre el peor y el mejor caso: {pesos(u.canasta_peor - u.canasta_optima)}/mes")

### ¿Qué productos explican el ahorro?

No todos pesan igual. Si el 70% del ahorro está en 5 productos, el consejo
práctico es "fijate el precio de estos cinco", no "recorré todo el supermercado".

In [ ]:
ap = (ult.merge(dim[["item", "cantidad_mes"]], on="item")
         .assign(ahorro=lambda x: (x.precio_med - x.precio_min) * x.cantidad_mes)
         .groupby("item")["ahorro"].mean()
         .sort_values(ascending=False))
ap_pct = 100 * ap.cumsum() / ap.sum()
display(pd.DataFrame({"ahorro_mensual": ap.round(0), "acumulado_pct": ap_pct.round(1)}).head(15))

n5 = ap_pct.iloc[4]
print(f"\nLos 5 productos con más ahorro concentran el {n5:.0f}% del total.")

## Hallazgo 3 — Ranking de cadenas

Índice 100 = mediana del mercado, en la misma provincia y el mismo día.

Uso **mediana de ratios** y no ratio de medianas: así una cadena con surtido
distinto (más productos caros en su mix) no distorsiona el índice.

In [ ]:
r = (rank[rank.fecha == rank.fecha.max()]
        .groupby("cadena")
        .agg(indice=("indice_vs_mercado", "median"),
             items=("items", "sum"), provincias=("provincia", "nunique"))
        .sort_values("indice"))
display(r.round(1))

print(f"Entre la más barata y la más cara hay {r.indice.max()-r.indice.min():.0f} puntos.")
print("Ojo: un índice bajo no implica que convenga siempre. La sección siguiente lo matiza.")

### ¿La más barata lo es en todo?

Pregunta que un entrevistador va a hacer. Si ninguna cadena gana en todas las
categorías, el consejo "comprá siempre en X" es incorrecto y el dashboard —que
permite mirar producto por producto— gana sentido.

In [ ]:
pi = precio[precio.fecha == precio.fecha.max()]
ref = pi.groupby(["item", "provincia"])["precio_mediano"].transform("median")
pi = pi.assign(ratio=100 * pi.precio_mediano / ref)

tabla = pi.pivot_table(index="cadena", columns="categoria", values="ratio", aggfunc="median").round(0)
try:                                   # el degradado necesita jinja2
    display(tabla.style.background_gradient(cmap="RdYlGn_r", axis=None).format("{:.0f}"))
except (AttributeError, ImportError):
    display(tabla)

ganadores = tabla.idxmin()
print("\nCadena más barata por categoría:")
print(ganadores.to_string())
print(f"\nCadenas distintas que lideran alguna categoría: {ganadores.nunique()}")

## Hallazgo 4 — Geografía

In [ ]:
g = (canast[canast.fecha == canast.fecha.max()]
        .groupby("provincia")["canasta_tipica"].mean().sort_values())
display(g.round(0))
print(f"\nBrecha entre la provincia más cara y la más barata: {100*(g.max()/g.min()-1):.1f}%")

ax = g.plot.barh(color="#1f4e79")
ax.axvline(g.mean(), color="#c0392b", ls="--", lw=1, label="Promedio")
ax.set_xlabel("Costo mensual de la canasta ($)"); ax.legend(frameon=False)
ax.set_title("Costo de la canasta por provincia");

## Hallazgo 5 — ¿Alguna cadena mueve los precios primero?

Con solo 7 días esto es **exploratorio, no concluyente**, y hay que decirlo así.
Pero es la pregunta que abre la puerta al proyecto siguiente: correr el pipeline
a diario durante meses y ver quién lidera los aumentos.

In [ ]:
ser = (precio.groupby(["fecha", "cadena"])["precio_mediano"].median().unstack())
var = (100 * ser.pct_change()).dropna(how="all")
display(var.round(2))

ax = var.plot(marker="o", figsize=(11, 4.5))
ax.axhline(0, color="#333", lw=.8)
ax.set_ylabel("Variación diaria (%)"); ax.set_title("Variación diaria del precio mediano, por cadena")
ax.legend(fontsize=8, ncol=3, frameon=False)
print("Con 7 días no se puede concluir liderazgo de precios. Sirve para plantear la hipótesis.")

## Las tres frases del pitch

Completá con tus números y practicá decirlas en voz alta. Si no podés completar
las tres, el proyecto no está terminado.

1. *"El mismo producto, en la misma provincia, cuesta hasta ___% más caro según
   la cadena. Comparado por kilo o por litro, no por precio de envase."*

2. *"Para un hogar de 4 personas eso son $______ por mes, un ___% de la canasta.
   Y el ___% de ese ahorro está en solo 5 productos."*

3. *"La cadena más barata está ___ puntos por debajo de la mediana del mercado y
   la más cara ___ puntos por encima, pero ninguna gana en todas las categorías."*

### Las cuatro preguntas que te van a hacer

**"¿Por qué mediana y no promedio?"**
Una cadena concentra el ___% de las sucursales. El promedio terminaría siendo su
precio disfrazado de precio de mercado.

**"¿Cómo sabés que comparás el mismo producto?"**
No lo sé con certeza. Por eso normalizo a precio por kilo o litro, exijo que la
unidad coincida con la esperada del item, y descarto lo que se aleja demasiado de
la referencia. Es la principal limitación y está declarada en el README.

**"¿Esto contradice al INDEC?"**
No. Mi canasta pondera distinto, cubre solo grandes superficies y usa precios de
lista sin promociones. Son cosas distintas y no son comparables.

**"¿Qué harías con más tiempo?"**
Orquestar la ingesta diaria con alertas cuando una cadena deja de reportar,
tests de calidad declarativos, matcheo de productos por embeddings en vez de
patrones de texto, y una serie de 12 meses para contrastar con estacionalidad.